# Data Preparation & EDA — Transactions Internationales

Ce notebook couvre :
1. **Data Cleaning** — nettoyage, types, valeurs manquantes, doublons, outliers
2. **Encoding** — transformation des variables catégorielles
3. **Feature Engineering** — indicateurs métier dérivés
4. **EDA** — exploration statistique et visuelle

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Chargement des données

In [ ]:
df = pd.read_csv("transactions.csv")
print(f"Shape : {df.shape}")
df.head(3)

In [ ]:
df.dtypes

In [ ]:
df.describe(include="all").T

## 2. Data Cleaning

**Pourquoi :** un modèle ou une analyse fiable dépend de données propres. Sans nettoyage, on introduit du bruit, des erreurs d'interprétation et des graphiques trompeurs.

### 2.1 Conversion des dates

In [ ]:
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
print(f"Dates invalides (NaT) : {df['transaction_date'].isna().sum()}")
df["transaction_date"].dtype

### 2.2 Suppression des doublons

In [ ]:
n_before = len(df)
df = df.drop_duplicates()
print(f"Doublons supprimés : {n_before - len(df)}")

### 2.3 Vérification et correction des types

In [ ]:
num_cols = [
    "amount", "exchange_rate_to_MAD", "amount_MAD",
    "company_age", "transaction_frequency", "avg_transaction_amount",
    "partner_concentration_index", "risk_score",
    "notation_pays", "score_risque_pays", "risque_operationnel",
    "score_aml", "rang_transaction_entreprise",
    "montant_precedent", "evolution_montant_pct"
]

cat_cols = [
    "currency", "origin_country", "destination_country",
    "trade_region", "company_sector", "company_size",
    "company_city", "payment_method", "transaction_type"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print("Types numériques après conversion :")
df[num_cols].dtypes

### 2.4 Valeurs manquantes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"missing": missing, "pct": missing_pct})
missing_df[missing_df["missing"] > 0].sort_values("pct", ascending=False)

In [ ]:
# Imputation : médiane pour les numériques, 'Unknown' pour les catégorielles
for col in num_cols:
    if col in df.columns and df[col].isna().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

for col in cat_cols:
    if col in df.columns and df[col].isna().sum() > 0:
        df[col] = df[col].fillna("Unknown")

print(f"Valeurs manquantes restantes : {df.isnull().sum().sum()}")

### 2.5 Détection des outliers extrêmes (IQR)

In [ ]:
outlier_cols = ["amount_MAD", "amount", "evolution_montant_pct"]

for col in outlier_cols:
    if col not in df.columns:
        continue
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 3 * IQR
    upper = Q3 + 3 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_out} outliers extrêmes ({n_out/len(df)*100:.2f}%) — borne [{lower:.0f}, {upper:.0f}]")

## 3. Encoding

**Pourquoi :** les algorithmes ML ne comprennent pas le texte directement. L'encodage transforme les catégories en variables numériques exploitables.

In [ ]:
categorical_cols_to_encode = [
    "company_city",
    "company_sector",
    "currency",
    "payment_method",
    "transaction_type",
    "trade_region",
    "company_size"
]

# Utilise get_dummies (OneHotEncoding), drop_first=True évite la multicolinéarité
encoded = pd.get_dummies(
    df[[c for c in categorical_cols_to_encode if c in df.columns]],
    drop_first=True
)

df_model = pd.concat(
    [df.drop(columns=[c for c in categorical_cols_to_encode if c in df.columns]), encoded],
    axis=1
)

print(f"Shape avant encoding : {df.shape}")
print(f"Shape après encoding  : {df_model.shape}")
encoded.head(2)

## 4. Feature Engineering

**Pourquoi :** transformer des données brutes en indicateurs métier utiles pour le clustering, le scoring ou la détection d'anomalies.

### 4.1 Volume total par entreprise

In [ ]:
volume_by_company = df.groupby("company_id")["amount_MAD"].sum().rename("total_volume_MAD")
df = df.merge(volume_by_company, on="company_id", how="left")
print("total_volume_MAD — statistiques :")
df["total_volume_MAD"].describe()

### 4.2 Nombre de transactions par entreprise

In [ ]:
freq_by_company = df.groupby("company_id")["transaction_id"].count().rename("tx_count")
df = df.merge(freq_by_company, on="company_id", how="left")
df["tx_count"].describe()

### 4.3 Pays partenaire et diversification internationale

In [ ]:
df["partner_country"] = df.apply(
    lambda x: x["origin_country"] if x["transaction_type"] == "import" else x["destination_country"],
    axis=1
)

partner_count = (
    df.groupby("company_id")["partner_country"]
    .nunique()
    .rename("n_partner_countries")
)
df = df.merge(partner_count, on="company_id", how="left")
df["n_partner_countries"].describe()

### 4.4 Balance commerciale import/export

In [ ]:
imports = (
    df[df["transaction_type"] == "import"]
    .groupby("company_id")["amount_MAD"].sum()
)
exports = (
    df[df["transaction_type"] == "export"]
    .groupby("company_id")["amount_MAD"].sum()
)

balance = pd.concat(
    [imports.rename("total_imports"), exports.rename("total_exports")],
    axis=1
).fillna(0)

balance["trade_balance"] = balance["total_exports"] - balance["total_imports"]
balance["trade_balance_ratio"] = (
    balance["total_exports"] / (balance["total_exports"] + balance["total_imports"] + 1e-9)
)

df = df.merge(balance, on="company_id", how="left")
balance.describe()

### 4.5 Tendances mensuelles — extraction date

In [ ]:
df["year"] = df["transaction_date"].dt.year
df["month"] = df["transaction_date"].dt.month
df["year_month"] = df["transaction_date"].dt.to_period("M")

print("Colonnes temporelles ajoutées : year, month, year_month")
df[["transaction_date", "year", "month", "year_month"]].head(3)

## 5. EDA — Analyse Exploratoire

**Pourquoi :** comprendre le comportement commercial avant toute modélisation (clustering, scoring, détection d'anomalies).

### 5.1 Top 10 pays partenaires

In [ ]:
top_countries = df["partner_country"].value_counts().head(10)

fig, ax = plt.subplots()
top_countries.plot(kind="bar", ax=ax, color=sns.color_palette("muted", 10))
ax.set_title("Top 10 pays partenaires")
ax.set_xlabel("Pays")
ax.set_ylabel("Nombre de transactions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

top_countries

### 5.2 Distribution des devises

In [ ]:
currency_dist = df["currency"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

currency_dist.plot(kind="bar", ax=axes[0], color=sns.color_palette("pastel"))
axes[0].set_title("Distribution des devises (%)")
axes[0].set_ylabel("%")
axes[0].tick_params(axis="x", rotation=0)

axes[1].pie(
    currency_dist,
    labels=currency_dist.index,
    autopct="%1.1f%%",
    colors=sns.color_palette("pastel")
)
axes[1].set_title("Répartition des devises")

plt.tight_layout()
plt.show()

currency_dist.round(2)

### 5.3 Types de transactions et modes de paiement

In [ ]:
tx_type_dist = df["transaction_type"].value_counts(normalize=True) * 100
payment_dist = df["payment_method"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(data=df, x="transaction_type", ax=axes[0], palette="muted")
axes[0].set_title("Types de transactions")
axes[0].set_xlabel("")

sns.countplot(data=df, x="payment_method", ax=axes[1], palette="muted")
axes[1].set_title("Modes de paiement")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

print("Types (%) :\n", tx_type_dist.round(2))
print("\nPaiements (%) :\n", payment_dist.round(2))

### 5.4 Tendance mensuelle du volume (amount_MAD)

In [ ]:
monthly_trend = df.groupby("year_month")["amount_MAD"].sum().reset_index()
monthly_trend["year_month"] = monthly_trend["year_month"].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(
    monthly_trend["year_month"],
    monthly_trend["amount_MAD"] / 1e9,
    marker="o",
    linewidth=1.5,
    markersize=3,
    color="steelblue"
)
ax.set_title("Volume mensuel total (en milliards MAD)")
ax.set_xlabel("Mois")
ax.set_ylabel("Milliards MAD")
step = max(1, len(monthly_trend) // 12)
ax.set_xticks(range(0, len(monthly_trend), step))
ax.set_xticklabels(monthly_trend["year_month"].iloc[::step], rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.5 Boxplot — amount_MAD par secteur

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(
    data=df,
    x="company_sector",
    y="amount_MAD",
    palette="muted",
    ax=ax,
    showfliers=False  # masque les outliers extrêmes pour la lisibilité
)
ax.set_title("Distribution du montant (MAD) par secteur")
ax.set_xlabel("Secteur")
ax.set_ylabel("Amount MAD")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

### 5.6 Boxplot — amount_MAD par type de transaction

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(
    data=df,
    x="transaction_type",
    y="amount_MAD",
    palette="muted",
    ax=ax,
    showfliers=False
)
ax.set_title("Distribution du montant (MAD) par type de transaction")
ax.set_xlabel("")
ax.set_ylabel("Amount MAD")
plt.tight_layout()
plt.show()

### 5.7 Matrice de corrélation des variables numériques clés

In [ ]:
corr_cols = [
    "amount_MAD", "risk_score", "score_aml", "score_risque_pays",
    "risque_operationnel", "company_age", "transaction_frequency",
    "partner_concentration_index", "n_partner_countries",
    "total_volume_MAD", "tx_count", "trade_balance_ratio"
]
corr_cols = [c for c in corr_cols if c in df.columns]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
    square=True,
    linewidths=0.5
)
ax.set_title("Matrice de corrélation — variables numériques clés")
plt.tight_layout()
plt.show()

## 6. Export du dataset nettoyé et enrichi

In [ ]:
# Colonnes à exclure de l'export (temporaires ou redondantes)
cols_to_drop = ["year_month"]  # Period n'est pas sérialisable en CSV tel quel

df_export = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
df_export.to_csv("transactions_prepared.csv", index=False)

print(f"Fichier exporté : transactions_prepared.csv")
print(f"Shape finale : {df_export.shape}")
df_export.head(3)

## Résumé du pipeline

| Étape | Action | Résultat |
|---|---|---|
| Cleaning | Conversion dates, types, doublons, NaN, outliers | Données fiables |
| Encoding | `get_dummies` sur 7 colonnes catégorielles | Variables numériques pour ML |
| Feature Engineering | volume, tx_count, partenaires, balance | Indicateurs métier |
| EDA | 7 graphiques, matrice de corrélation | Compréhension des flux |
| Export | `transactions_prepared.csv` | Dataset prêt pour modélisation |